# Lab 07 · Frameworks and evaluation

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## The same small workflow, two ways

**Goal:** run a real LangGraph graph, compare it with plain Python, and measure more than whether code completed.

Our workflow classifies a support message, then chooses a short reply. Two nodes are enough to see what a framework adds: explicit state and edges. This is a small workflow, not a claim of full autonomy.

This lab uses the actual `langgraph` package, pinned to `1.0.2` for reproducibility. Installation may use the network; practice model calls still require no API key. The earlier plain-Python examples need no framework; the protocol labs use their own SDKs.

In [ ]:
import importlib.metadata
import subprocess
import sys
try:
    installed = importlib.metadata.version("langgraph")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "1.0.2":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.0.2"])
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
print("LangGraph:", importlib.metadata.version("langgraph"))

## Define two small steps

The first step calls the model. The second is ordinary Python. The practice fixture is selected from a visible lookup table; those labels are not passed to a live model. For a new practice message, add its label to that table. A missing fixture becomes `unknown` and is rejected visibly.

In [ ]:
practice_labels = {"My parcel is late.": "delivery", "I forgot my password.": "account",
                   "Where is my delivery?": "delivery"}
replies = {"delivery": "Please share the tracking number.",
           "account": "Please share the login error message."}

def classify(state):
    label = client.ask("Return only delivery or account for: " + state["message"],
        practice=practice_labels.get(state["message"], "unknown")).strip().lower()
    if label not in replies:
        raise ValueError("Unknown category; ask a human")
    return {"category": label}

In [ ]:
def write_reply(state):
    return {"reply": replies[state["category"]]}

def plain_python(message):
    state = {"message": message}
    state.update(classify(state))
    state.update(write_reply(state))
    return state

plain_result = plain_python("My parcel is late.")
print(plain_result)
assert plain_result["reply"].strip()

## Build a real two-node graph

A node reads state and returns an update. Edges specify the execution order. `compile()` builds the runnable graph; `invoke()` executes it.

In [ ]:
class SupportState(TypedDict, total=False):
    message: str
    category: str
    reply: str

builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("reply", write_reply)
builder.add_edge(START, "classify")
builder.add_edge("classify", "reply")
builder.add_edge("reply", END)
graph = builder.compile()
graph_result = graph.invoke({"message": "My parcel is late."})
print(graph_result)
assert graph_result["reply"].strip()

**Expected practice result:** both versions choose `delivery` and ask for a tracking number. Live calls may differ, even with the same prompt. A framework does not make model output deterministic or correct.

## A tiny evaluation

Use two new messages with expected categories. They are held out from the worked input above. The fixtures encode their labels, so practice scores check the evaluation wiring only. They are not empirical model accuracy.

In [ ]:
import time
cases = [("I forgot my password.", "account"),
         ("Where is my delivery?", "delivery")]
records = []
for method, run in [("python", plain_python),
                    ("langgraph", lambda text: graph.invoke({"message": text}))]:
    for message, expected in cases:
        before, started = len(client.trace), time.perf_counter()
        output = run(message)
        records.append({"method": method, "correct": output["category"] == expected,
            "calls": len(client.trace) - before, "seconds": time.perf_counter() - started})
print(records)
assert len(records) == 4 and all(r["calls"] == 1 for r in records)

In [ ]:
for method in ["python", "langgraph"]:
    rows = [r for r in records if r["method"] == method]
    print(method, {"accuracy": sum(r["correct"] for r in rows) / len(rows),
        "calls": sum(r["calls"] for r in rows),
        "wall_seconds": round(sum(r["seconds"] for r in rows), 4)})
print("Reported API usage:", summarize_trace([client]))
if MODE == "practice":
    print("Scripted examples only: these scores do not measure model quality.")

Two examples are far too few for a general accuracy claim. A latency difference here may be noise. Both methods make one model call per message; the framework handles orchestration, not fewer tokens. Unknown usage or monetary cost must stay unknown.

**Tiny exercises:**

1. Add one ambiguous support message and decide when a human should review it.
2. Add a third formatting node and compare it with one extra Python function call.
3. In live mode, inspect wrong labels and reply usefulness. Freeze prompts before testing; tuning on the test examples leaks information.

For a larger evaluation, separate development and test data, include failures and abstentions, rate factual support and usefulness, and record calls, tokens, cost when known, and latency. A passing shape check alone is not a quality metric.

## Frameworks beyond this example

LangGraph provides explicit state and graph control. Other frameworks offer different abstractions, such as higher-level agents or role-based teams. Choose based on the behavior you need, rather than the number of agents a framework can launch.

## Read further

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [Pinned LangGraph release](https://pypi.org/project/langgraph/1.0.2/)
- [HELM evaluation](https://crfm.stanford.edu/helm/)

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.